In [1]:
from IPython.core.display import HTML
import random, os, json

import spacy
from spacy.tokens import DocBin
from spacy.training import Example

/home/ronji/Dev/HRAI/spacy_cnn_ner_en/env/lib/python3.14/site-packages/confection/__init__.py:38: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel, Extra, ValidationError, create_model


ConfigError: unable to infer type for attribute "REGEX"

In [4]:
nlp = spacy.load(os.getcwd() + 'spacy_outputs/model-last')

In [5]:
ruler = nlp.add_pipe('entity_ruler').from_disk(os.getcwd() + 'patterns.jsonl') # type: ignore

In [8]:
doc_bin = DocBin().from_disk(os.getcwd() +'spacy_outputs/test.spacy')
docs = list(doc_bin.get_docs(nlp.vocab))

examples = []
for doc in docs:
    example = Example.from_dict(doc, {"entities": [(ent.start_char, ent.end_char, ent.label_) for ent in doc.ents]})
    examples.append(example)

results = nlp.evaluate(examples)
print(f"""
    precision: {results['ents_p']:.2f}
        = kolik entit, které model označil jako „pozitivní“, byly skutečně správně
    recall: {results['ents_r']:.2f}"
        = Kolika všech skutečných „pozitivních“ případů si model ve výsledku všiml
    F1: {results['ents_f']:.2f}
        = harmonický průměr precision a recall; pro snažší určení výkonnosti
    soft skills/schopnosti: 
    {results['ents_per_type']['Soft Skill']}

    hard skills/dovednosti: 
    {results['ents_per_type']['Hard Skill']}
""")


    precision: 0.56
        = kolik entit, které model označil jako „pozitivní“, byly skutečně správně
    recall: 1.00"
        = Kolika všech skutečných „pozitivních“ případů si model ve výsledku všiml
    F1: 0.71
        = harmonický průměr precision a recall; pro snažší určení výkonnosti
    soft skills/schopnosti: 
    {'p': 0.4743886743886744, 'r': 1.0, 'f': 0.6435055865921787}

    hard skills/dovednosti: 
    {'p': 0.5967505659874817, 'r': 1.0, 'f': 0.7474562135112593}



In [9]:
def render_ents(text, ents): # substitude for spacy display not working with latest python version -- at time of making
    colors = {'Hard Skill': '#cfffdc','Soft Skill': '#68ba7f','Certification':'#253d2c'}
    entities = sorted(ents, key=lambda x: x.start_char) # sort by spacy ent start index
    # build jupyter html 
    html = ''
    last_idx = 0
    for e in ents:
        start, end, label = e.start_char, e.end_char, e.label_
        html +=text[last_idx:start] # normal
        html += f'<mark style="background-color: {colors[label]}">{text[start:end]}</mark>'
        last_idx = end
    html += text[last_idx:]
    display(HTML(html))

In [10]:
# random file for demo

with open(os.getcwd() + 'data/resumes_en.json','r') as f:
    docs = json.loads(f.read())

i = random.randrange(2450)
text = docs[i]
doc = nlp(text)

s = "Soft skills:\n"
h = "Hard skills: \n"
for e in doc.ents:
    if e.label_ =="Soft Skill":
        s = s + e.text + ", "
    elif e.label_ =="Hard Skill":
        h = h + e.text + ", "
print(s+"\n\n"+h)

render_ents(text,doc.ents)

Soft skills:
Editing, Research, Research, construction, planning, Planning, Research, Editing, Research, 

Hard skills: 
MATLAB, Simulink, LATEX, AutoCAD, Laser, Statistics, STATA, Navicat, Operating Systems, aerodynamics, economics, data analysis, Navicat, SQLite, STATA, chassis, assembly line, industrial production, Mechatronics, Robotics, Arduino, assembly language, Robotics, Kinematics, trajectory, arm, painting, Rendering, Mechanical Engineering, AutoCAD, motion planning, trajectory planning, algorithms, Trajectory, Kalman Filter, Extended Kalman Filter, Computer Vision, geometry, optical flow, RANSAC, cartography, optical flow, Kalman Filter, Feedback Control, block diagram, Frequency Response, Machine Learning, Supervised Learning, Unsupervised Learning, Logistic Regression, Adaboost, algorithms, TDD, Matlab, simulations, Applied Science, Applied Mechanics, Applied Mechanics, Automation, Automation, assembly language, AutoCAD, data analysis, economics, Laser, Machine Learning, M